**System prompts vs. user messages** is just applying Chapter 1's split, now that the prompt has real content in it.
- The **system prompt** holds everything that stays true no matter which email comes in: the class definitions, the domain rules, the examples — the constant "job description"
- The **user message** stays the one variable thing per call: `Subject: ... / Body: ...`
- As the system prompt grows from a one-liner to a few hundred words of domain knowledge, the user message never changes shape — that separation is exactly why versioning the system prompt is safe to do in isolation

**Zero-shot classification** is the v0 you already built — no examples, no extra domain knowledge, just the three label names. It's not a placeholder to throw away; it's the **floor** every later improvement gets measured against.
- It works because the label names themselves (`FD`, `Non-FD`) carry real meaning the model already has from pretraining
- It fails quietly on exactly the cases your MuRIL error analysis already flagged: ambiguous short emails, and anything needing the FD-reference-number pattern to disambiguate
- Worth keeping as `v0` in the registry forever, even after you move past it — it's your honest baseline number

**Injecting domain knowledge** means putting what your rule engine already learned about this data *into the prompt*, instead of making the model rediscover it from scratch.
- The **FD reference pattern** (`BJ` + year + `FD` + digits) is a near-deterministic signal your classical pipeline already relies on — give it to the model explicitly rather than hoping it infers the pattern
- The **keyword groups** and **negation list** are the other half — but a negated FD keyword (*"paisa nahi aaya"* — money hasn't come) usually still means the email **is** about FD, just phrased as a complaint, so the prompt has to say that explicitly or the model will under-weight negated mentions
- I used **placeholder** groups in the code below, not your real 8 groups / 20-phrase list — swap those in before treating any comparison as real

**Few-shot prompting** means showing the model a handful of real, labeled examples instead of just describing the categories.
- Each example should be a genuine row from your dataset, formatted exactly like the output you want back — that's also free training data for the output format itself
- **"How many is enough?"** — in practice, **one example per class** is usually the inflection point; going from 0→3 examples (covering every class once) typically helps far more than going from 3→10. More examples mostly help once you start adding examples of specific *failure modes*, not just one-per-class coverage
- The code below uses exactly 3 — one per class — and deliberately **not** the row used as the test query, so the comparison isn't testing on an example the model already saw

**Chain-of-thought prompting** asks the model to reason through the evidence *before* committing to a label, instead of jumping straight to an answer.
- The instruction alone ("reason step by step") doesn't *guarantee* the order — what actually enforces it here is the **schema**: a `reasoning` field declared first and marked required is emitted before `label`, because structured outputs preserve schema order for required fields
- This is the one version that costs meaningfully more **output tokens** — worth paying for only on the cases v0–v2 are actually getting wrong, not as a blanket upgrade
- It also gives you something to read when the model is wrong — the `reasoning` field is your debug log, not just an accuracy lever

**Structured output** has quietly changed since Chapter 1 — Anthropic now has a way to *guarantee* schema-valid JSON, instead of just asking nicely and stripping markdown fences after the fact.
- `client.messages.parse(..., output_format=YourPydanticModel)` uses **constrained decoding** — the model is structurally restricted to only ever produce tokens that satisfy your schema
- The payoff: `response.parsed_output` is already a real Pydantic object — no `json.loads()`, no fence-stripping, no `PARSE_ERROR` fallback needed anywhere in the code below
- The two cases it *can't* fully guarantee: a safety **refusal** (`stop_reason="refusal"`) and a response cut short by **`max_tokens`** — both still worth checking for, just far rarer than the markdown-fence problem from before

**Prompt versioning** means treating each prompt edit like a code change — what changed, and why you expect it to help — not just overwriting the same variable.
- The `PROMPT_REGISTRY` below pairs every prompt with a **changelog** entry: not just the text, but the hypothesis behind the change
- `log_run()` appends every actual classification — version, input, predicted label, true label — to a local file, so by the time you reach Chapter 5's evaluation, you already have the raw data to diff `v0` against `v3` instead of re-running everything from scratch
- This is the same discipline as your classical pipeline's hyperparameter sweep logs — just applied to prompt text instead of learning rate and batch size

In [1]:
import os
import json
from datetime import datetime, timezone
from typing import Literal
 
import pandas as pd
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from anthropic import Anthropic

import anthropic

load_dotenv()  # reads .env in the current working directory
api_key = os.getenv("anthropic_api_key")
client = anthropic.Anthropic(api_key=api_key)
MODEL_ID = "claude-haiku-4-5-20251001"

In [2]:
class EmailClassification(BaseModel):
    """Used by v0, v1, v2."""
    label: Literal["FD", "Non-FD", "Multiple Category"]
    reason: str
 
 
class EmailClassificationWithReasoning(BaseModel):
    """Used by v3 (chain-of-thought). `reasoning` is declared FIRST and is
    required, so the model is structurally forced to generate it BEFORE
    `label` -- per Anthropic's docs, required properties are emitted in
    schema order. That ordering is what makes this genuine chain-of-thought,
    not just an extra field bolted on after the decision is already made."""
    reasoning: str = Field(description="Step-by-step reasoning, written BEFORE deciding the label.")
    label: Literal["FD", "Non-FD", "Multiple Category"]
    reason: str = Field(description="One short sentence summarizing the final decision.")

In [3]:
# ----------------------------------------------------------------------
# Sub-topic 1: system prompt vs. user message.
# Everything below is the constant "job description" (system). The one
# variable input per call is built fresh in classify_email() (user message).
# ----------------------------------------------------------------------
 
# Sub-topic 2: v0 -- zero-shot, no domain knowledge (Chapter 1's baseline)

V0_ZERO_SHOT = """You classify customer emails for Bajaj Finance into one of three categories:
 
- FD: the email is about a Fixed Deposit account
- Non-FD: the email is about anything else
- Multiple Category: the email is about both FD and non-FD topics
"""

In [4]:
# Sub-topic 3: v1 -- domain knowledge injected.
# NOTE: the keyword groups and negation phrases below are ILLUSTRATIVE
# placeholders. Swap in your actual 8 keyword groups and 20-phrase negation
# list from the rule-engine cheatsheet before treating this as a real
# comparison against the classical baseline.

V1_DOMAIN_KNOWLEDGE = """You classify customer emails for Bajaj Finance into one of three categories:
 
- FD: the email is about a Fixed Deposit account -- maturity, interest/quarterly
  payout, premature withdrawal, rollover, or an FD reference number matching the
  pattern BJ + 4-digit year + FD + digits (e.g. BJ2019FD7717).
- Non-FD: the email is about anything else -- loans, EMIs, insurance, cards, the
  mobile app, or branch service, with no FD reference.
- Multiple Category: the email raises BOTH an FD concern AND a Non-FD concern.
 
Domain signal groups:
- FD-signal keywords : maturity, interest payout, premature withdrawal, rollover, FD reference number
- Non-FD-signal keywords : EMI, loan, insurance premium, card, app login, branch visit
- Negation phrases to watch for: "nahi mila" (didn't get), "abhi tak nahi aaya"
  (still hasn't come) -- a negated FD keyword can still mean the email IS about
  FD (the customer is complaining something FD-related didn't happen), so
  don't auto-route purely on keyword presence.
 
Emails are often in Hinglish (Hindi written in Roman script, mixed with English).
Treat Hinglish phrasing as normal customer language, not as noise.
"""

In [5]:
# Sub-topic 4: v2 -- few-shot. Real examples pulled from the dataset (rows
# 803, 1215, 1757) -- deliberately NOT the row used as the test query below,
# so we're not testing on an example the model was already shown.

V2_FEW_SHOT = V1_DOMAIN_KNOWLEDGE + """
Examples:
 
Email: "Mera paisa abhi tak nahi aaya. Kab milega? Bahut time ho gaya hai. BJ2024FD9354."
Output: {"label": "FD", "reason": "Customer is asking about money tied to FD reference BJ2024FD9354."}
 
Email: "Sir ji, App me login nahi ho raha. OTP aata hi nahi. Teen din se try kar raha hoon. Kya problem hai?"
Output: {"label": "Non-FD", "reason": "Complaint is about app login/OTP, no FD reference at all."}
 
Email: "Aapke yahan mera paisa hai BJ2022FD5397. Uska kya status hai? And separately, I want to foreclose my personal loan BJ2023CD2320."
Output: {"label": "Multiple Category", "reason": "Raises both an FD status question and a separate loan foreclosure request."}
"""

In [6]:
# Sub-topic 5: v3 -- chain-of-thought. Same domain knowledge as v1, plus an
# explicit instruction to reason before deciding. The instruction alone
# wouldn't guarantee ordering -- EmailClassificationWithReasoning's field
# order is what actually enforces "reason, then decide".

V3_CHAIN_OF_THOUGHT = V1_DOMAIN_KNOWLEDGE + """
Before deciding, reason step by step: list every FD-signal and every
Non-FD-signal phrase you find in the email, check each one for negation,
then commit to a label based on that reasoning.
"""

In [ ]:
# ----------------------------------------------------------------------
# Sub-topic 7: prompt versioning. The registry tracks not just the prompt
# text, but WHAT changed and WHY -- the part a raw string variable can't.
# ----------------------------------------------------------------------
PROMPT_REGISTRY = {
    "v0_zero_shot": {
        "system_prompt": V0_ZERO_SHOT,
        "schema": EmailClassification,
        "max_tokens": 200,
        "changelog": "Baseline. No domain knowledge, no examples. What's the floor?",
    },
    "v1_domain_knowledge": {
        "system_prompt": V1_DOMAIN_KNOWLEDGE,
        "schema": EmailClassification,
        "max_tokens": 200,
        "changelog": "Added FD reference pattern, keyword groups, negation warning, "
                      "Hinglish note. Expect fewer keyword-only misreads than v0.",
    },
    "v2_few_shot": {
        "system_prompt": V2_FEW_SHOT,
        "schema": EmailClassification,
        "max_tokens": 200,
        "changelog": "Added 3 real labeled examples (one per class) on top of v1. "
                      "Expect Multiple Category to improve most -- it's the hardest "
                      "class to describe in rules alone.",
    },
    "v3_chain_of_thought": {
        "system_prompt": V3_CHAIN_OF_THOUGHT,
        "schema": EmailClassificationWithReasoning,
        "max_tokens": 600,
        "changelog": "Same domain knowledge as v1, but forces explicit reasoning "
                      "before the label via schema field ordering. Costs more output "
                      "tokens -- only worth it if v0-v2 get borderline cases wrong.",
    },
}

In [8]:
def classify_email(version: str, subject: str, content: str):
    """Run one email through a named prompt version. Returns the parsed
    Pydantic object directly -- no json.loads(), no fence-stripping, no
    PARSE_ERROR fallback. Structured outputs guarantee schema-valid output."""
    entry = PROMPT_REGISTRY[version]
    user_message = f"Subject: {subject}\n\nBody: {content}"
 
    response = client.messages.parse(
        model=MODEL_ID,
        max_tokens=300,
        system=entry["system_prompt"],
        messages=[{"role": "user", "content": user_message}],
        output_format=entry["schema"],
    )
    return response.parsed_output
 

In [9]:
def log_run(version: str, subject: str, true_label: str, result, log_path: str = "prompt_run_log.jsonl"):
    """Append one run to a local log -- the practical half of versioning:
    not just what the prompt says, but what it actually produced, when,
    and under which version. Makes Chapter 5's eval comparisons possible."""
    entry = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "version": version,
        "subject": subject,
        "true_label": true_label,
        "predicted_label": result.label,
        "model_reason": result.reason,
    }
    with open(log_path, "a") as f:
        f.write(json.dumps(entry) + "\n")

In [10]:
if __name__ == "__main__":
    df = pd.read_csv("../data/fd_dataset_messy.csv")
    sample = df.iloc[5]  # Shobha Chopra, BJ2019FD7717, true label = FD
 
    print("=" * 70)
    print("TEST EMAIL (same one across every version, for a fair comparison)")
    print("=" * 70)
    print(f"Subject : {sample['subject']}")
    print(f"Content : {sample['content']}")
    print(f"True label : {sample['label']}")
 
    for version in PROMPT_REGISTRY:
        entry = PROMPT_REGISTRY[version]
        print("\n" + "-" * 70)
        print(f"VERSION: {version}")
        print(f"Changelog: {entry['changelog']}")
        print("-" * 70)
 
        result = classify_email(version, sample["subject"], sample["content"])
 
        if hasattr(result, "reasoning"):
            print(f"Reasoning (CoT) : {result.reasoning}")
        print(f"Predicted label : {result.label}")
        print(f"Reason          : {result.reason}")
 
        log_run(version, sample["subject"], sample["label"], result)
 
    print("\n" + "=" * 70)
    print("All 4 runs logged to prompt_run_log.jsonl -- one line per version, "
          "ready to diff across iterations.")
    print("=" * 70)

TEST EMAIL (same one across every version, for a fair comparison)
Subject : Help
Content : Hello ji, Yeh second baar likh raha hoon. My funds with your company is stuck. The period was over in December but nothing came to my bank. Please check BJ2019FD7717. Jaldi kuch karo please. Thank you. Shobha Chopra | 9686667744
True label : FD

----------------------------------------------------------------------
VERSION: v0_zero_shot
Changelog: Baseline. No domain knowledge, no examples. What's the floor?
----------------------------------------------------------------------
Predicted label : FD
Reason          : The email is about a Fixed Deposit account issue. The customer mentions 'My funds with your company is stuck', references a period that ended in December, and provides an FD account number (BJ2019FD7717) starting with 'FD'. The complaint is specifically about their FD maturity and non-receipt of funds.

----------------------------------------------------------------------
VERSION: v1

ValidationError: 1 validation error for EmailClassificationWithReasoning
  Invalid JSON: EOF while parsing a string at line 1 column 1008 [type=json_invalid, input_value='{"reasoning":"Step-by-st...d Deposit (BJ2019FD7717', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/json_invalid